In [2]:
import sympy as sp
import sympy.physics.mechanics as me 
sp.init_printing(use_latex="mathjax")
me.init_vprinting()
from IPython.display import display

In [3]:
# Frames, points, coordinates declaration

N, B, U1, U2, L1, L2 = sp.symbols('N, B, U_1, U_2, L_1, L_2', cls=me.ReferenceFrame)
O, T, C1, C2, R1, R2 = sp.symbols('O, T, C_1, C_2, R_1, R_2', cls=me.Point)                                 # CoM points for the bodies
A1, A2, B1, B2, K1, K2, O1, O2 = sp.symbols('A_1, A_2, B_1, B_2, K_1, K_2, O_1, O_2', cls=me.Point)         # points for constraints / loop closure
theta1, theta2 = me.dynamicsymbols('q1 q2', real="True")
alpha, beta = me.dynamicsymbols('q3 q4', real="True")
lt, ls, lc, lr, lf = sp.symbols('l_t, l_s, l_c, l_r, l_f' , positive="True", real="True")
m, g = sp.symbols('m, g', real="True", positive="True")
t = me.dynamicsymbols._t


p = sp.Matrix([ lt, ls, lc, lr, lf, m, g])
u1, u2, u3, u4 = me.dynamicsymbols('u1:5')
q = sp.Matrix([theta1, theta2])
qr = sp.Matrix([alpha, beta])
qN = q.col_join(qr)

u = sp.Matrix([u1, u2])
ur = sp.Matrix([u3, u4])
uN = u.col_join(ur)

qdN = qN.diff(t)
ud = u.diff(t)

qn_zero = {q:0 for q in qN}
ur_zero = {u: 0 for u in ur}
uN_zero = {u: 0 for u in uN}
qdN_zero = {qd: 0 for qd in qdN}
ud_zero = {ud: 0 for ud in ud}

q, qr, qN, u, ur, uN, qdN, ud

⎛            ⎡q₁⎤              ⎡u₁⎤  ⎡q₁̇⎤      ⎞
⎜            ⎢  ⎥              ⎢  ⎥  ⎢  ⎥      ⎟
⎜⎡q₁⎤  ⎡q₃⎤  ⎢q₂⎥  ⎡u₁⎤  ⎡u₃⎤  ⎢u₂⎥  ⎢q₂̇⎥  ⎡u₁̇⎤⎟
⎜⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥⎟
⎜⎣q₂⎦  ⎣q₄⎦  ⎢q₃⎥  ⎣u₂⎦  ⎣u₄⎦  ⎢u₃⎥  ⎢q₃̇⎥  ⎣u₂̇⎦⎟
⎜            ⎢  ⎥              ⎢  ⎥  ⎢  ⎥      ⎟
⎝            ⎣q₄⎦              ⎣u₄⎦  ⎣q₄̇⎦      ⎠

In [4]:
# Framing and coordinates setting

B.orient_body_fixed(N, (alpha, beta, 0), 'XYZ')
U1.orient_axis(B, B.y, theta1)
U2.orient_axis(B, B.y, theta2)

T.set_pos(O, lt*B.z)
O1.set_pos(O, lc*N.x + ls/2*N.y - lf*N.z)
O2.set_pos(O, lc*N.x - ls/2*N.y - lf*N.z)

A1.set_pos(O, ls/2*U1.y + (lr-lf)*U1.z)
C1.set_pos(O, lc/2*U1.x + ls/2*U1.y + (lr-lf)*U1.z)
B1.set_pos(C1, lc/2*U1.x)

A2.set_pos(O,- ls/2*U2.y + (lr-lf)*U2.z)
C2.set_pos(O, lc/2*U2.x - ls/2*U2.y + (lr-lf)*U2.z)
B2.set_pos(C2, lc/2*U2.x)


In [5]:
# position check

O1.pos_from(O).express(N).xreplace(qn_zero)

In [6]:
me.find_dynamicsymbols(T.pos_from(O), reference_frame=N), me.find_dynamicsymbols(C1.pos_from(O), reference_frame=N), me.find_dynamicsymbols(C2.pos_from(O), reference_frame=N)

## Holonomic constraints

In [7]:
# loop closure equations

rod1_constraint = O1.pos_from(O) - B1.pos_from(O)
rod2_constraint = O2.pos_from(O) - B2.pos_from(O)

constraints = [
    rod1_constraint,
    rod2_constraint
    ]

for c in constraints:
    display(c.express(N))


In [8]:
# Holonomic constraints

fh = sp.Matrix([
    sp.trigsimp(rod1_constraint.dot(rod1_constraint)) - lr**2,
    sp.trigsimp(rod2_constraint.dot(rod2_constraint)) - lr**2,
    ])

fh.simplify()        


In [9]:
fh

⎡                                                                              ↪
⎢       2                     2                                                ↪
⎢- 2⋅l_c ⋅cos(q₁ + q₄) + 2⋅l_c  - 2⋅l_c⋅l_f⋅sin(q₁ + q₄)⋅cos(q₃) + 2⋅l_c⋅l_f⋅s ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢       2                     2                                                ↪
⎢- 2⋅l_c ⋅cos(q₂ + q₄) + 2⋅l_c  - 2⋅l_c⋅l_f⋅sin(q₂ + q₄)⋅cos(q₃) + 2⋅l_c⋅l_f⋅s ↪
⎣                                                                              ↪

↪                                                                              ↪
↪                                                                          2   ↪
↪ in(q₁ + q₄) - 2⋅l_c⋅lᵣ⋅sin(q₁ + q₄) - l_c⋅lₛ⋅sin(q₁ + q₄)⋅sin(q₃) - 2⋅l_f ⋅c ↪
↪                          

In [10]:
me.find_dynamicsymbols(fh)

## Kinematic diffential equations

In [11]:
fk = sp.Matrix([
    theta1.diff() - u1,
    theta2.diff() - u2,
    alpha.diff() - u3,
    beta.diff() - u4
])

In [12]:
Mk = fk.jacobian(qdN)
gk = fk.xreplace(qdN_zero)

qdN_sol = -Mk.LUsolve(gk)
qdN_sol


⎡u₁⎤
⎢  ⎥
⎢u₂⎥
⎢  ⎥
⎢u₃⎥
⎢  ⎥
⎣u₄⎦

In [13]:
qd_repl = dict(zip(qdN, qdN_sol))
qd_repl

In [14]:
fhd = fh.diff(t)
me.find_dynamicsymbols(fhd)

In [15]:
fhd = fhd.xreplace(qd_repl)
fhd = sp.trigsimp(fhd)
fhd

⎡                                                                              ↪
⎢     2                                                                        ↪
⎢2⋅l_c ⋅(u₁ + u₄)⋅sin(q₁ + q₄) - 2⋅l_c⋅l_f⋅(u₁ + u₄)⋅cos(q₁ + q₄)⋅cos(q₃) + 2⋅ ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢     2                                                                        ↪
⎢2⋅l_c ⋅(u₂ + u₄)⋅sin(q₂ + q₄) - 2⋅l_c⋅l_f⋅(u₂ + u₄)⋅cos(q₂ + q₄)⋅cos(q₃) + 2⋅ ↪
⎣                                                                              ↪

↪                                                                              ↪
↪                                                                              ↪
↪ l_c⋅l_f⋅(u₁ + u₄)⋅cos(q₁ + q₄) + 2⋅l_c⋅l_f⋅u₃⋅sin(q₁ + q₄)⋅sin(q₃) - 2⋅l_c⋅l ↪
↪                          

In [16]:
me.find_dynamicsymbols(fhd)

In [17]:
Mhd = fhd.jacobian(ur)
ghd = fhd.xreplace(ur_zero)
ur_sol = -Mhd.LUsolve(ghd)

ur_repl = dict(zip(ur,ur_sol))
me.find_dynamicsymbols(ur_sol)